# Phase 5: Automated scoring pipeline

**Goal:** run GR00T N1.7 and Cosmos-Reason2-2B across a batch of episodes, then apply
`docs/scoring_rubric.md`'s match/failure rules to what they produced — all in this one
notebook, top to bottom. Earlier drafts split this into a GPU notebook plus a separate
local scoring script, but that just meant syncing files between two places for no real
benefit — the scoring math and the reasoning-stage LLM judge don't need a GPU, but they
don't need to be *off* Colab either, so it's simpler to keep everything in one place you
run once.

Sections 0-11 collect raw data (GR00T's predicted actions, the actual recorded actions,
Cosmos-Reason2's reasoning trace) and write it to Drive as JSON, one file per episode.
Sections 12-16 read that JSON back and score it: execution-stage tiers per body-part
group, a Claude API call as the reasoning-stage judge, and the core reasoning x execution
2x2 classification. See `docs/scoring_rubric.md` for the full rule definitions.

**Design note — three observation points per episode, not one.** The Phase 3 smoke test
queried GR00T once, at frame 0. To get a real per-phase reasoning-vs-execution
comparison (the project's core "intent lost in handoff" 2x2 in the rubric), GR00T is
queried **three times per episode** here: once at the start of each derived sub-goal
(reach → transport → retreat, using the grasp/release frame indices from the hand-closure
segmentation below). Cosmos-Reason2 still runs only **once** per episode, since it
produces one full step-by-step plan up front, not a new plan per phase. This roughly
triples GR00T's per-episode compute cost but is what actually lets the scoring stage
tell whether a good plan and a good hand-off held up through transport and retreat, not
just at the very first step.

**If any cell errors: stop, copy the full error output, and paste it back.** Same rule
as notebook 1 — don't try to patch it yourself.

## 0. Drive mount + checkpoint scaffolding

Separate checkpoint file and raw-log directory from notebook 1 (`phase5_status.json`,
`raw_logs/`), so this can be re-run independently without touching Phase 2/3's status.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time

PROJECT_DIR = '/content/drive/MyDrive/humanoid-vla-eval'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'g1_teleop_subset')
RAW_LOG_DIR = os.path.join(PROJECT_DIR, 'raw_logs')
STATUS_PATH = os.path.join(CKPT_DIR, 'phase5_status.json')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_LOG_DIR, exist_ok=True)

def load_status():
    if os.path.exists(STATUS_PATH):
        with open(STATUS_PATH) as f:
            return json.load(f)
    return {}

def save_status(key, value):
    status = load_status()
    status[key] = value
    status['_last_updated'] = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(STATUS_PATH, 'w') as f:
        json.dump(status, f, indent=2)
    return status

def is_done(key):
    return load_status().get(key) == 'done'

print('Current status:', load_status())

## 1. Install dependencies

Identical to notebook 1's install cell — copied verbatim so this notebook can run in a
fresh Colab session on its own, without depending on notebook 1 having run first in the
same kernel.

In [ ]:
import os, subprocess, platform, importlib, sys

def run(cmd):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')

GR00T_REPO_PATH = '/content/Isaac-GR00T'
RESTART_MARKER = '/content/.numpy_clean_reinstall_done'

def clean_reinstall_numpy():
    result = subprocess.run(
        [sys.executable, '-c', 'import numpy, os; print(os.path.dirname(numpy.__file__))'],
        capture_output=True, text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        numpy_dir = result.stdout.strip()
        print(f'Found existing numpy install at: {numpy_dir} — removing it entirely.')
        run(f'rm -rf "{numpy_dir}"')
    else:
        print('No existing numpy import found (or already removed) — nothing to clean.')
    run(f'"{sys.executable}" -m pip uninstall -y numpy')
    run(f'uv pip install --python "{sys.executable}" --reinstall --no-cache numpy==1.26.4')
    with open(RESTART_MARKER, 'w') as f:
        f.write('numpy was reinstalled cleanly on disk; the CURRENT process may still have a '
                'stale copy loaded and needs a restart to pick it up.')

def gr00t_deep_import():
    if os.path.isdir(GR00T_REPO_PATH) and GR00T_REPO_PATH not in sys.path:
        sys.path.insert(0, GR00T_REPO_PATH)
    importlib.invalidate_caches()
    try:
        import gr00t  # noqa: F401
        import gr00t.policy  # noqa: F401
        import gr00t.data.embodiment_tags  # noqa: F401
        import pandas  # noqa: F401
        import pyarrow  # noqa: F401
        import cv2  # noqa: F401
        import pytorch_kinematics  # noqa: F401
        return True, False
    except ImportError:
        return False, False
    except ValueError as e:
        print(f'numpy ABI mismatch in this process: {e}')
        return False, True

def print_diagnostics():
    print('\n--- DIAGNOSTICS ---')
    print('Kernel sys.executable:', sys.executable)
    print()
    subprocess.run(f'"{sys.executable}" -m pip show gr00t numpy pyarrow pytorch-kinematics', shell=True)
    print()
    print('Fresh-subprocess import test (same interpreter as kernel):')
    result = subprocess.run([sys.executable, '-c', 'import gr00t.policy, pandas, pyarrow, cv2, pytorch_kinematics; import numpy; print("OK:", gr00t.__file__, numpy.__version__)'], capture_output=True, text=True)
    print('returncode:', result.returncode)
    print('stdout:', result.stdout)
    print('stderr:', result.stderr)

if os.path.exists(RESTART_MARKER):
    os.remove(RESTART_MARKER)

ok, needs_restart = gr00t_deep_import()

if ok:
    print('gr00t already importable — skipping install.')
    save_status('deps_installed', 'done')
elif needs_restart:
    print('numpy is already loaded in this running kernel in a way that cannot be fixed '
          'without a restart. Cleaning the on-disk install now so a restart picks up a '
          'consistent state immediately...')
    clean_reinstall_numpy()
    raise RuntimeError(
        'ACTION REQUIRED: Runtime > Restart session (NOT "Restart and run all" — just '
        'restart), then re-run this cell.'
    )
else:
    if is_done('deps_installed'):
        print("Checkpoint says deps_installed=done but the deep import fails — reinstalling for real.")

    py_ver = platform.python_version()
    print(f'Python version: {py_ver}')
    if not py_ver.startswith('3.12'):
        print('WARNING: Isaac-GR00T requires Python 3.12.x.')

    if not os.path.exists(GR00T_REPO_PATH):
        run(f'git clone https://github.com/NVIDIA/Isaac-GR00T.git {GR00T_REPO_PATH}')

    run('pip install -q -U uv')
    run(f'uv pip install --python "{sys.executable}" -e {GR00T_REPO_PATH}')
    run(f'uv pip install --python "{sys.executable}" accelerate pyarrow pytorch-kinematics')

    ok, needs_restart = gr00t_deep_import()
    if needs_restart:
        clean_reinstall_numpy()
        raise RuntimeError('ACTION REQUIRED: Runtime > Restart session, then re-run this cell.')
    elif not ok:
        print_diagnostics()
        raise RuntimeError(
            'Install commands reported success but the deep import still fails. Copy the '
            'full output of this cell back to Claude.'
        )

    save_status('deps_installed', 'done')
    print('Dependencies installed and verified importable.')

## 2. Hugging Face auth

In [ ]:
from huggingface_hub import login
login()

## 3. Load GR00T N1.7 (System 1 / action model)

In [ ]:
import torch

GR00T_CHECKPOINT = 'nvidia/GR00T-N1.7-3B'

from gr00t.policy import Gr00tPolicy
from gr00t.data.embodiment_tags import EmbodimentTag

gr00t_policy = Gr00tPolicy(
    model_path=GR00T_CHECKPOINT,
    embodiment_tag=EmbodimentTag.REAL_G1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
gr00t_modality_configs = gr00t_policy.get_modality_config()
print('GR00T N1.7 loaded successfully with EmbodimentTag.REAL_G1.')
for modality, cfg in gr00t_modality_configs.items():
    print(f'{modality}: keys={cfg.modality_keys}, horizon={len(cfg.delta_indices)}')
save_status('gr00t_loaded', 'done')

## 4. Load Cosmos-Reason2-2B (System 2 reasoning proxy)

In [ ]:
COSMOS_REASON2_CHECKPOINT = 'nvidia/Cosmos-Reason2-2B'

import transformers

PIXELS_PER_TOKEN = 32 ** 2

cosmos_model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(
    COSMOS_REASON2_CHECKPOINT,
    dtype=torch.float16,
    device_map='auto',
    attn_implementation='sdpa',
)
cosmos_processor = transformers.Qwen3VLProcessor.from_pretrained(COSMOS_REASON2_CHECKPOINT)

min_vision_tokens, max_vision_tokens = 256, 8192
cosmos_processor.image_processor.size = {
    'shortest_edge': min_vision_tokens * PIXELS_PER_TOKEN,
    'longest_edge': max_vision_tokens * PIXELS_PER_TOKEN,
}
cosmos_processor.video_processor.size = {
    'shortest_edge': min_vision_tokens * PIXELS_PER_TOKEN,
    'longest_edge': max_vision_tokens * PIXELS_PER_TOKEN,
}
print('Cosmos-Reason2-2B loaded successfully.')
save_status('cosmos_loaded', 'done')

## 5. Dataset

Reuses the same `g1-pick-apple` episode subset from Phase 2/3. `snapshot_download`
caches by content, so if this points at the same `DATA_DIR` as notebook 1 already used,
nothing gets re-downloaded — this just re-derives the file list. Bump `N_EPISODES` here
if you want to score more episodes than Phase 2 originally pulled.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
import json as _json

G1_DATASET_REPO = 'nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1'
TASK_FOLDER = 'g1-pick-apple'
N_EPISODES = 50  # bumped from 15 (2026-07-20) -- deadline-driven scale-up, see PROJECT_STATUS.md
TASK_DIR = os.path.join(DATA_DIR, TASK_FOLDER)

if not is_done('g1_meta_downloaded'):
    snapshot_download(
        repo_id=G1_DATASET_REPO, repo_type='dataset',
        allow_patterns=[f'{TASK_FOLDER}/meta/*'], local_dir=DATA_DIR,
    )
    save_status('g1_meta_downloaded', 'done')

with open(os.path.join(TASK_DIR, 'meta', 'info.json')) as f:
    info = _json.load(f)

discarded = set(info.get('discarded_episode_indices', []))
total_episodes = info['total_episodes']

selected_episodes = []
i = 0
while len(selected_episodes) < N_EPISODES and i < total_episodes:
    if i not in discarded:
        selected_episodes.append(i)
    i += 1
print(f'Selected {len(selected_episodes)} episodes:', selected_episodes)

allow_patterns = []
for ep in selected_episodes:
    allow_patterns.append(f'{TASK_FOLDER}/data/chunk-000/episode_{ep:06d}.parquet')
    allow_patterns.append(f'{TASK_FOLDER}/videos/chunk-000/observation.images.ego_view/episode_{ep:06d}.mp4')
snapshot_download(
    repo_id=G1_DATASET_REPO, repo_type='dataset',
    allow_patterns=allow_patterns, local_dir=DATA_DIR,
)
save_status('g1_episodes_downloaded', 'done')

with open(os.path.join(TASK_DIR, 'meta', 'modality.json')) as f:
    modality_spec = _json.load(f)

episodes_meta = {}
with open(os.path.join(TASK_DIR, 'meta', 'episodes.jsonl')) as f:
    for line in f:
        rec = _json.loads(line)
        episodes_meta[rec['episode_index']] = rec

print('Ready:', len(selected_episodes), 'episodes in', TASK_DIR)

## 6. Shared parsing + forward-kinematics helpers

Same functions as notebook 1 Sections 6/6.5 — copied here rather than imported, since a
notebook can't reliably `import` from another `.ipynb`. Keep these two copies in sync by
hand if either changes.

In [ ]:
import pandas as pd
import numpy as np
import cv2
import urllib.request
import pytorch_kinematics as pk

def split_by_modality(vec, spec):
    vec = np.asarray(vec, dtype=np.float32)
    return {name: vec[bounds['start']:bounds['end']] for name, bounds in spec.items()}

G1_URDF_URL = 'https://raw.githubusercontent.com/unitreerobotics/unitree_ros/master/robots/g1_description/g1_29dof_with_hand_rev_1_0.urdf'
G1_URDF_PATH = os.path.join(CKPT_DIR, 'g1_29dof_with_hand_rev_1_0.urdf')

if not os.path.exists(G1_URDF_PATH):
    urllib.request.urlretrieve(G1_URDF_URL, G1_URDF_PATH)
with open(G1_URDF_PATH, 'rb') as f:
    urdf_bytes = f.read()

left_wrist_chain = pk.build_serial_chain_from_urdf(urdf_bytes, 'left_wrist_yaw_link')
right_wrist_chain = pk.build_serial_chain_from_urdf(urdf_bytes, 'right_wrist_yaw_link')

raw_joint_names = info['features']['observation.state']['names']
assert len(raw_joint_names) == 43

def rot_matrix_to_rot6d(R):
    return R[:, :2].T.reshape(-1)

def compute_wrist_eef_9d(chain, raw_state_vec):
    joint_name_to_val = dict(zip(raw_joint_names, raw_state_vec))
    needed_names = chain.get_joint_parameter_names()
    th = [float(joint_name_to_val[name]) for name in needed_names]
    ret = chain.forward_kinematics(th, end_only=False)
    end_link_name = list(ret.keys())[-1]
    m = ret[end_link_name].get_matrix().numpy()[0]
    pos = m[:3, 3]
    rot6d = rot_matrix_to_rot6d(m[:3, :3])
    return np.concatenate([pos, rot6d]).astype(np.float32)

FK_CHAINS = {'left_wrist_eef_9d': left_wrist_chain, 'right_wrist_eef_9d': right_wrist_chain}
FK_KEYS = set(FK_CHAINS.keys())
print('FK helpers ready.')

## 7. Ground-truth sub-goal segmentation

Implements `docs/scoring_rubric.md` Section 2: derive reach/transport/retreat frame
boundaries from a hand-closure signal (mean abs value of the 7 hand-joint state dims per
frame), rather than from any labeled sub-goal annotation the dataset doesn't have. The
threshold is fit from each episode's own signal range, not a fixed hardcoded value. If
no clean grasp+release pair is found, or the resulting phases are wildly unbalanced,
`segmentation_valid=False` and the episode is limited to a reach-only observation point
(handled in Section 9) — matches the rubric's Section 6 manual-review trigger for a bad
segmentation.

In [ ]:
def compute_hand_closure_signal(df, modality_spec, hand_key):
    bounds = modality_spec['state'][hand_key]
    n_frames = len(df)
    signal = np.zeros(n_frames, dtype=np.float64)
    for t in range(n_frames):
        vec = np.asarray(df.iloc[t]['observation.state'], dtype=np.float64)
        signal[t] = np.mean(np.abs(vec[bounds['start']:bounds['end']]))
    return signal

def fit_closure_threshold(signal):
    """Midpoint-of-range threshold, fit per-episode rather than hardcoded — the exact
    fraction (0.4) is a starting point per scoring_rubric.md Section 2 and is exactly the
    kind of numeric cutoff that section says to revisit after the manual spot-check, not
    a value we're claiming is precisely correct yet."""
    return float(signal.min() + 0.4 * (signal.max() - signal.min()))

def find_events(signal, threshold, min_dwell=3):
    above = signal > threshold
    n = len(above)
    grasp_idx = None
    i = 0
    while i < n:
        if above[i] and i + min_dwell <= n and np.all(above[i:i + min_dwell]):
            grasp_idx = i
            break
        i += 1
    if grasp_idx is None:
        return None, None
    release_idx = None
    i = grasp_idx
    while i < n:
        if not above[i] and i + min_dwell <= n and np.all(~above[i:i + min_dwell]):
            release_idx = i
            break
        i += 1
    return grasp_idx, release_idx

def derive_subgoal_segments(df, modality_spec):
    left_signal = compute_hand_closure_signal(df, modality_spec, 'left_hand')
    right_signal = compute_hand_closure_signal(df, modality_spec, 'right_hand')
    left_range = left_signal.max() - left_signal.min()
    right_range = right_signal.max() - right_signal.min()
    if left_range >= right_range:
        combined_signal, active_hand = left_signal, 'left_hand'
    else:
        combined_signal, active_hand = right_signal, 'right_hand'

    threshold = fit_closure_threshold(combined_signal)
    grasp_idx, release_idx = find_events(combined_signal, threshold)
    n_frames = len(df)
    valid = grasp_idx is not None and release_idx is not None
    if valid:
        reach_frac = grasp_idx / n_frames
        transport_frac = (release_idx - grasp_idx) / n_frames
        retreat_frac = (n_frames - release_idx) / n_frames
        if min(reach_frac, transport_frac, retreat_frac) < 0.05:
            valid = False

    return {
        'active_hand': active_hand,
        'threshold': threshold,
        'signal_min': float(combined_signal.min()),
        'signal_max': float(combined_signal.max()),
        'grasp_idx': grasp_idx,
        'release_idx': release_idx,
        'n_frames': n_frames,
        'segmentation_valid': valid,
        'hand_closure_signal': combined_signal.tolist(),
    }

print('Segmentation helpers ready.')

## 8. Build an observation at an arbitrary start frame

Generalizes notebook 1 Section 7 (which always built the observation from frame 0) to
start from any frame index — needed so GR00T can be queried at the transport and retreat
sub-goal starts, not just the episode start. Same key-discovery pattern as before
(reads `gr00t_modality_configs` rather than hardcoding key names), plus a `cv2` seek to
the requested start frame.

In [ ]:
def build_observation(df, video_path, episode_length, start_frame, task_instruction,
                       gr00t_modality_configs, modality_spec, fk_chains, fk_keys):
    video_keys = gr00t_modality_configs['video'].modality_keys
    state_keys = gr00t_modality_configs['state'].modality_keys
    language_keys = gr00t_modality_configs['language'].modality_keys
    T_video = len(gr00t_modality_configs['video'].delta_indices)
    T_state = len(gr00t_modality_configs['state'].delta_indices)
    assert len(video_keys) == 1 and len(language_keys) == 1

    # Sequential read-and-discard rather than cap.set(CAP_PROP_POS_FRAMES, ...): for
    # H.264-encoded video, a direct seek can snap to the nearest keyframe instead of the
    # exact requested frame, which would desync the video from the state row pulled at
    # the same index below. Episodes here are short (~100 frames), so decoding up to
    # start_frame sequentially is cheap and guarantees frame-exact alignment -- this
    # matters for the transport/retreat observation points (start_frame != 0), not the
    # reach point (start_frame == 0, where a seek is a no-op anyway).
    cap = cv2.VideoCapture(video_path)
    for _ in range(start_frame):
        if not cap.read()[0]:
            break
    frames = []
    for _ in range(T_video):
        ret, f = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    while 0 < len(frames) < T_video:
        frames.append(frames[-1])
    video_arr = np.stack(frames, axis=0)[np.newaxis, ...]
    observation_video = {video_keys[0]: video_arr}

    raw_state_rows = []
    for t in range(start_frame, start_frame + T_state):
        idx = min(t, episode_length - 1)
        raw_state_rows.append(np.asarray(df.iloc[idx]['observation.state'], dtype=np.float64))
    state_split_rows = [split_by_modality(row, modality_spec['state']) for row in raw_state_rows]

    observation_state = {}
    for k in state_keys:
        if k in fk_keys:
            vals = np.stack([compute_wrist_eef_9d(fk_chains[k], row) for row in raw_state_rows], axis=0)
        else:
            vals = np.stack([row[k] for row in state_split_rows], axis=0)
        observation_state[k] = vals[np.newaxis, ...]

    observation_language = {language_keys[0]: [[task_instruction]]}
    return {'video': observation_video, 'state': observation_state, 'language': observation_language}

print('build_observation() ready.')

## 9. Per-episode pipeline

Runs Cosmos-Reason2 once (frame 0, full task) and GR00T at each valid observation point
(reach/transport/retreat), and packages the raw predicted-vs-actual data the scoring
script needs. No scoring judgment happens here — just data collection.

In [ ]:
from PIL import Image
import tempfile

def get_actual_action_window(df, episode_length, start_frame, T_action, modality_spec, fk_chains, fk_keys):
    end_frame = min(start_frame + T_action, episode_length)
    actual = {k: [] for k in list(modality_spec['action'].keys()) + list(fk_keys)}
    for t in range(start_frame, end_frame):
        vec = np.asarray(df.iloc[t]['action'], dtype=np.float64)
        split = split_by_modality(vec, modality_spec['action'])
        for k, v in split.items():
            actual[k].append(v.tolist())
        for k in fk_keys:
            actual[k].append(compute_wrist_eef_9d(fk_chains[k], vec).tolist())
    return actual, end_frame - start_frame

def run_episode(EP, task_folder, task_dir, episodes_meta, modality_spec,
                 gr00t_policy, gr00t_modality_configs, cosmos_model, cosmos_processor,
                 fk_chains, fk_keys):
    parquet_path = os.path.join(task_dir, 'data', 'chunk-000', f'episode_{EP:06d}.parquet')
    video_path = os.path.join(task_dir, 'videos', 'chunk-000', 'observation.images.ego_view', f'episode_{EP:06d}.mp4')
    df = pd.read_parquet(parquet_path)
    episode_length = episodes_meta[EP]['length']
    task_instruction = episodes_meta[EP]['tasks'][0]

    segments = derive_subgoal_segments(df, modality_spec)

    cap = cv2.VideoCapture(video_path)
    ret, frame_bgr = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f'Could not read frame 0 of episode {EP} video: {video_path}')
    first_frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    tmp_frame_path = os.path.join(tempfile.gettempdir(), f'cosmos_scoring_ep{EP}.png')
    Image.fromarray(first_frame_rgb).save(tmp_frame_path)

    reasoning_prompt = (
        f'Task: "{task_instruction}". Describe the sub-goals needed to complete this task, '
        f'in order, as a short numbered list.'
    )
    conversation = [
        {'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': tmp_frame_path},
            {'type': 'text', 'text': reasoning_prompt},
        ]},
    ]
    cosmos_inputs = cosmos_processor.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt',
    ).to(cosmos_model.device)
    with torch.no_grad():
        cosmos_out_ids = cosmos_model.generate(**cosmos_inputs, max_new_tokens=200)
    cosmos_out_trimmed = cosmos_out_ids[:, cosmos_inputs['input_ids'].shape[1]:]
    cosmos_reasoning_text = cosmos_processor.batch_decode(
        cosmos_out_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False,
    )[0]

    observation_points = [('reach', 0)]
    if segments['segmentation_valid']:
        observation_points.append(('transport', segments['grasp_idx']))
        observation_points.append(('retreat', segments['release_idx']))

    phase_results = {}
    for phase_name, start_frame in observation_points:
        observation = build_observation(
            df, video_path, episode_length, start_frame, task_instruction,
            gr00t_modality_configs, modality_spec, fk_chains, fk_keys,
        )
        gr00t_action, _ = gr00t_policy.get_action(observation)
        T_action = next(iter(gr00t_action.values())).shape[1]
        mismatched = {k: v.shape[1] for k, v in gr00t_action.items() if v.shape[1] != T_action}
        if mismatched:
            raise ValueError(
                f'Episode {EP}, phase {phase_name}: GR00T returned different action '
                f'horizons across keys ({mismatched}, expected {T_action} for all) -- '
                f'stop and paste this back, do not silently truncate.'
            )
        actual_action, actual_len = get_actual_action_window(
            df, episode_length, start_frame, T_action, modality_spec, fk_chains, fk_keys,
        )
        predicted_action = {k: v[0, :actual_len, :].tolist() for k, v in gr00t_action.items()}
        phase_results[phase_name] = {
            'start_frame': start_frame,
            'T_action': T_action,
            'compared_steps': actual_len,
            'predicted_action': predicted_action,
            'actual_action': actual_action,
        }

    return {
        'task_folder': task_folder,
        'episode_index': EP,
        'task_instruction': task_instruction,
        'episode_length': episode_length,
        'segmentation': segments,
        'cosmos_reasoning_proxy_text': cosmos_reasoning_text,
        'phase_results': phase_results,
        'wrist_eef_9d_note': (
            "left_wrist_eef_9d / right_wrist_eef_9d computed via forward kinematics "
            "(public Unitree G1 URDF); approximation, not verified against NVIDIA's "
            "internal training convention."
        ),
    }

print('run_episode() ready.')

## 10. Main loop — run across selected episodes

Checkpointed per episode: an episode whose JSON already exists in `RAW_LOG_DIR` is
skipped, so a disconnected session resumes cleanly without redoing finished episodes.

In [ ]:
for EP in selected_episodes:
    out_path = os.path.join(RAW_LOG_DIR, f'episode_{EP:06d}.json')
    if os.path.exists(out_path):
        print(f'Episode {EP}: already logged, skipping.')
        continue
    print(f'Episode {EP}: running...')
    result = run_episode(
        EP, TASK_FOLDER, TASK_DIR, episodes_meta, modality_spec,
        gr00t_policy, gr00t_modality_configs, cosmos_model, cosmos_processor,
        FK_CHAINS, FK_KEYS,
    )
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'Episode {EP}: done. segmentation_valid={result["segmentation"]["segmentation_valid"]}, '
          f'active_hand={result["segmentation"]["active_hand"]}, '
          f'phases_logged={list(result["phase_results"].keys())}')

save_status('scoring_pass_1', 'done')
print('\nAll episodes processed. Raw logs at:', RAW_LOG_DIR)

## 11. Status summary

Raw data collection is done once this prints a nonzero episode count. The scoring
sections below (12-16) read directly from `RAW_LOG_DIR` — no file syncing or separate
environment needed, just keep running cells in this same notebook.

In [ ]:
print(json.dumps(load_status(), indent=2))
n_logged = len([f for f in os.listdir(RAW_LOG_DIR) if f.endswith('.json')])
print(f'\n{n_logged} episode logs in {RAW_LOG_DIR}')

## 12. Scoring — install the Anthropic SDK + API key

Everything from here on runs on regular CPU (no GPU calls) — it reads the JSON files
already written to `RAW_LOG_DIR` and applies `docs/scoring_rubric.md`'s match/failure
rules to them, calling Claude once per episode as an LLM judge for the reasoning-stage
comparison. Kept in this same notebook rather than a separate script so the whole
pipeline runs top-to-bottom in one place, without syncing files or setting up a second
environment.

**Re-scoring already-collected data in a fresh session:** if `RAW_LOG_DIR` on Drive
already has episode JSON files from a previous run (e.g. after a scoring-logic fix, with
no need to re-run GR00T/Cosmos), you only need Section 0 (Drive mount — sets
`PROJECT_DIR`/`RAW_LOG_DIR`) plus Sections 12-16. Sections 1-11 (deps install, model
loading, dataset download, FK/segmentation helpers) are **not** needed — this section is
self-contained and imports its own `numpy`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'anthropic'], check=True)

import numpy as np  # scoring section only needs numpy, not the full gr00t/deps stack
import anthropic
import getpass

if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

anthropic_client = anthropic.Anthropic()
print('Anthropic client ready.')

## 13. Execution-stage scoring (`docs/scoring_rubric.md` Section 4)

Per body-part group: wrist position (cm) + rotation (geodesic degrees) for the two
`wrist_eef_9d` keys, per-joint degree diffs for arm/waist/leg keys, and a discrete
open/transitioning/closed state match for the two hand keys (reusing the same threshold
already fit for the ground-truth segmentation in Section 7).

In [ ]:
EXECUTION_ARM_WAIST_KEYS = {'left_arm', 'right_arm', 'waist', 'left_leg', 'right_leg'}
EXECUTION_HAND_KEYS = {'left_hand', 'right_hand'}
EXECUTION_WRIST_EEF_KEYS = {'left_wrist_eef_9d', 'right_wrist_eef_9d'}


def rot6d_to_matrix(rot6d):
    a1 = np.asarray(rot6d[0:3], dtype=np.float64)
    a2 = np.asarray(rot6d[3:6], dtype=np.float64)
    b1 = a1 / np.linalg.norm(a1)
    b2 = a2 - np.dot(b1, a2) * b1
    b2 = b2 / np.linalg.norm(b2)
    b3 = np.cross(b1, b2)
    return np.stack([b1, b2, b3], axis=1)


def geodesic_angle_deg(rot6d_a, rot6d_b):
    R_a = rot6d_to_matrix(rot6d_a)
    R_b = rot6d_to_matrix(rot6d_b)
    R_rel = R_a.T @ R_b
    trace = float(np.clip((np.trace(R_rel) - 1.0) / 2.0, -1.0, 1.0))
    return float(np.degrees(np.arccos(trace)))


def tier(value, match_max, minor_max):
    if value < match_max:
        return 'match'
    if value < minor_max:
        return 'minor_deviation'
    return 'failure'


def score_wrist_eef_key(predicted, actual):
    n = min(len(predicted), len(actual))
    if n == 0:
        return {'n_steps': 0, 'position_tier': 'unscored', 'rotation_tier': 'unscored'}
    pos_dists_cm, rot_angles_deg = [], []
    for t in range(n):
        pred, act = predicted[t], actual[t]
        pos_dist_m = float(np.linalg.norm(np.asarray(pred[:3]) - np.asarray(act[:3])))
        pos_dists_cm.append(pos_dist_m * 100.0)
        rot_angles_deg.append(geodesic_angle_deg(pred[3:9], act[3:9]))
    mean_pos_cm = sum(pos_dists_cm) / n
    mean_rot_deg = sum(rot_angles_deg) / n
    return {
        'n_steps': n,
        'mean_position_error_cm': round(mean_pos_cm, 3),
        'mean_rotation_error_deg': round(mean_rot_deg, 3),
        'position_tier': tier(mean_pos_cm, 3.0, 7.0),
        'rotation_tier': tier(mean_rot_deg, 10.0, 20.0),
    }


def score_joint_group(predicted, actual):
    """Assumes raw joint values are in radians (standard for robot joint-state
    representations) and converts to degrees for the rubric's thresholds — an
    assumption, not something independently confirmed against the dataset's units doc."""
    n = min(len(predicted), len(actual))
    if n == 0:
        return {'n_steps': 0, 'group_tier': 'unscored'}
    max_joint_diff_deg = 0.0
    mean_joint_diffs_deg = []
    for t in range(n):
        pred = np.asarray(predicted[t], dtype=np.float64)
        act = np.asarray(actual[t], dtype=np.float64)
        diff_deg = np.degrees(np.abs(pred - act))
        mean_joint_diffs_deg.append(diff_deg)
        max_joint_diff_deg = max(max_joint_diff_deg, float(diff_deg.max()))
    mean_per_joint = np.mean(np.stack(mean_joint_diffs_deg, axis=0), axis=0)
    if (mean_per_joint >= 15.0).any():
        group_tier = 'failure'
    elif (mean_per_joint >= 5.0).any():
        group_tier = 'minor_deviation'
    else:
        group_tier = 'match'
    return {
        'n_steps': n,
        'mean_per_joint_deg': [round(float(v), 3) for v in mean_per_joint],
        'max_single_joint_diff_deg': round(max_joint_diff_deg, 3),
        'group_tier': group_tier,
    }


def hand_state_bucket(mean_abs_value, threshold, signal_min, signal_max):
    band = 0.15 * (signal_max - signal_min)
    if mean_abs_value < threshold - band:
        return 'open'
    if mean_abs_value > threshold + band:
        return 'closed'
    return 'transitioning'


def score_hand_group(predicted, actual, threshold, signal_min, signal_max):
    n = min(len(predicted), len(actual))
    if n == 0:
        return {'n_steps': 0, 'group_tier': 'unscored'}
    matches = 0
    for t in range(n):
        pred_mag = float(np.mean(np.abs(predicted[t])))
        act_mag = float(np.mean(np.abs(actual[t])))
        if hand_state_bucket(pred_mag, threshold, signal_min, signal_max) == hand_state_bucket(act_mag, threshold, signal_min, signal_max):
            matches += 1
    match_frac = matches / n
    group_tier = 'match' if match_frac == 1.0 else ('minor_deviation' if match_frac >= 0.5 else 'failure')
    return {'n_steps': n, 'match_fraction': round(match_frac, 3), 'group_tier': group_tier}


def _tiers_for_keys(groups, keys):
    out = []
    for key in keys:
        g = groups.get(key)
        if g is None:
            continue
        if key in EXECUTION_WRIST_EEF_KEYS:
            out.append(g.get('position_tier', 'unscored'))
            out.append(g.get('rotation_tier', 'unscored'))
        else:
            out.append(g.get('group_tier', 'unscored'))
    return [t for t in out if t != 'unscored']


def _combine_tiers(tiers):
    if not tiers:
        return 'unscored'
    if all(t == 'match' for t in tiers):
        return 'match'
    if any(t == 'failure' for t in tiers):
        return 'failure'
    return 'minor_deviation'


EARLY_HORIZON_STEPS = 10  # ~0.5s at this dataset's 20fps. Checking signed (not just
                          # absolute) per-joint error on real data showed two distinct
                          # phenomena mixed together in a full-chunk average: some joints
                          # start near-zero error and grow steadily over the predicted
                          # horizon (normal open-loop drift, not evidence the plan wasn't
                          # followed), while others show a near-constant offset present
                          # even at step 0 (a more direct signal about the immediate
                          # predicted action). The match/failure verdict is scored on
                          # just the early window, matching how these policies are
                          # actually deployed (frequent re-planning, not a 40-step
                          # open-loop rollout) and isolating "did the immediate action
                          # follow the plan" from horizon-length drift. The full-chunk
                          # numbers are still computed and kept under
                          # `full_horizon_groups` for a secondary drift analysis.


def _score_groups(predicted, actual, segmentation):
    groups = {}
    for key in predicted:
        if key in EXECUTION_WRIST_EEF_KEYS:
            groups[key] = score_wrist_eef_key(predicted[key], actual.get(key, []))
        elif key in EXECUTION_HAND_KEYS:
            groups[key] = score_hand_group(
                predicted[key], actual.get(key, []),
                segmentation['threshold'], segmentation['signal_min'], segmentation['signal_max'],
            )
        elif key in EXECUTION_ARM_WAIST_KEYS:
            groups[key] = score_joint_group(predicted[key], actual.get(key, []))
    return groups


def score_execution_phase(phase_result, segmentation):
    """This is a single-handed pick-and-place task (see the active_hand note in
    Section 7) -- the arm/hand that ISN'T grasping isn't commanded to go anywhere in
    particular, so its prediction error is not evidence the plan wasn't executed. The
    overall phase verdict is driven by the active side (the arm/hand/wrist actually
    doing the grasping, per segmentation['active_hand']) plus the waist, which both
    sides' motion depends on. The idle side is still scored and reported in full for
    transparency -- it just doesn't drive the match/minor_deviation/failure
    classification. See EARLY_HORIZON_STEPS above for why classification uses only the
    first few predicted steps rather than the full chunk."""
    predicted = phase_result['predicted_action']
    actual = phase_result['actual_action']

    full_horizon_groups = _score_groups(predicted, actual, segmentation)

    early_predicted = {k: v[:EARLY_HORIZON_STEPS] for k, v in predicted.items()}
    early_actual = {k: v[:EARLY_HORIZON_STEPS] for k, v in actual.items()}
    groups = _score_groups(early_predicted, early_actual, segmentation)
    early_step_count = min(EARLY_HORIZON_STEPS, phase_result.get('compared_steps', 0))

    active_side = segmentation['active_hand'].split('_')[0]  # 'left' or 'right'
    idle_side = 'right' if active_side == 'left' else 'left'
    primary_keys = {f'{active_side}_arm', f'{active_side}_hand', f'{active_side}_wrist_eef_9d', 'waist'}
    idle_keys = {f'{idle_side}_arm', f'{idle_side}_hand', f'{idle_side}_wrist_eef_9d'}

    primary_tiers = _tiers_for_keys(groups, primary_keys)
    idle_tiers = _tiers_for_keys(groups, idle_keys)
    overall = _combine_tiers(primary_tiers)

    minor_count = sum(1 for t in primary_tiers if t == 'minor_deviation')
    manual_review = len(primary_tiers) > 0 and minor_count >= max(1, int(0.75 * len(primary_tiers)))

    return {
        'groups': groups,
        'full_horizon_groups': full_horizon_groups,
        'active_side': active_side,
        'primary_tiers': primary_tiers,
        'idle_side_tiers': idle_tiers,
        'overall_tier': overall,
        'early_step_count': early_step_count,
        'manual_review_ambiguous_minor': manual_review,
    }

print('Execution-stage scoring functions ready.')

## 14. Reasoning-stage scoring (`docs/scoring_rubric.md` Section 3) — LLM judge

One Claude call per episode: does Cosmos-Reason2's plan cover reach/transport/retreat,
in order, naming the right object and target, without hallucinating or being too vague
to map onto the three canonical phases? Uses structured output (`output_config.format`)
so the response is guaranteed valid JSON rather than parsed out of free text.

This is a small, cheap classification call — if per-call cost matters when scoring more
episodes than this batch, `claude-haiku-4-5` is a reasonable, much cheaper substitute for
`JUDGE_MODEL` below. Left as Opus by default rather than silently downgrading.

In [ ]:
JUDGE_MODEL = 'claude-opus-4-8'

REASONING_JUDGE_SCHEMA = {
    'type': 'object',
    'properties': {
        'object': {'type': 'string', 'description': 'The object being manipulated, as named in the task.'},
        'target': {'type': 'string', 'description': 'The destination/target, as named in the task.'},
        'phases_found': {
            'type': 'object',
            'properties': {
                'reach': {'type': 'boolean'},
                'transport': {'type': 'boolean'},
                'retreat': {'type': 'boolean'},
            },
            'required': ['reach', 'transport', 'retreat'],
            'additionalProperties': False,
        },
        'order_correct': {'type': 'boolean', 'description': 'True if the phases found appear in reach -> transport -> retreat order.'},
        'object_target_correct': {'type': 'boolean', 'description': 'True if every step referencing an object/target names the correct one from the task.'},
        'hallucinated_step': {'type': 'boolean', 'description': 'True if any step describes an action or object not grounded in the task.'},
        'under_specified': {'type': 'boolean', 'description': 'True if the plan is too vague to map onto any canonical phase confidently.'},
        'manual_review_needed': {'type': 'boolean', 'description': 'True if hallucinated_step or under_specified is true and you cannot confidently resolve it from the text alone.'},
        'rationale': {'type': 'string', 'description': 'One or two sentences explaining the judgment.'},
    },
    'required': [
        'object', 'target', 'phases_found', 'order_correct', 'object_target_correct',
        'hallucinated_step', 'under_specified', 'manual_review_needed', 'rationale',
    ],
    'additionalProperties': False,
}


def judge_reasoning(client, task_instruction, reasoning_text):
    prompt = (
        f'Task given to the robot: "{task_instruction}"\n\n'
        f'A model\'s step-by-step plan for this task:\n{reasoning_text}\n\n'
        'This is a pick-and-place task with exactly three canonical sub-goal phases: '
        'reach (approach and grasp the object), transport (carry the object toward the '
        'target/destination), and retreat (release the object and withdraw). Evaluate '
        'the plan above against these three phases.'
    )
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=1024,
        output_config={'format': {'type': 'json_schema', 'schema': REASONING_JUDGE_SCHEMA}},
        messages=[{'role': 'user', 'content': prompt}],
    )
    text = next(b.text for b in response.content if b.type == 'text')
    result = json.loads(text)

    all_phases_present = all(result['phases_found'].values())
    result['match'] = (
        all_phases_present and result['order_correct'] and result['object_target_correct']
        and not result['hallucinated_step'] and not result['under_specified']
    )
    failure_categories = []
    if not all_phases_present:
        failure_categories.append('missing_sub_goal')
    if all_phases_present and not result['order_correct']:
        failure_categories.append('wrong_order')
    if not result['object_target_correct']:
        failure_categories.append('wrong_object_target')
    if result['hallucinated_step']:
        failure_categories.append('hallucinated_step')
    if result['under_specified']:
        failure_categories.append('under_specified')
    result['failure_categories'] = failure_categories
    return result

print('Reasoning-stage judge ready.')

## 15. Core reasoning x execution classification (Section 5)

Cross-references the reasoning-stage phase match against the execution-stage tier for
that same phase — this is the actual research output the whole pipeline is built to
produce. `minor_deviation` execution results are tracked separately rather than folded
into match/failure, per the rubric.

In [ ]:
MIN_COMPARISON_STEPS = 3  # phases compared against fewer real frames than this
                          # (typically 'retreat' near the very end of a short episode)
                          # are too noisy to trust a match/failure verdict from -- flag
                          # for manual review instead of quietly classifying them.


def classify_2x2(reasoning_match_for_phase, execution_tier):
    if execution_tier == 'unscored':
        return 'unscored'  # not enough data to judge -- NOT the same as "failed"
    if execution_tier == 'minor_deviation':
        return 'minor_deviation'
    execution_match = execution_tier == 'match'
    if reasoning_match_for_phase and execution_match:
        return 'success'
    if reasoning_match_for_phase and not execution_match:
        return 'intent_lost_in_handoff'
    if not reasoning_match_for_phase and execution_match:
        return 'accidentally_correct'
    return 'compounding_failure'


def score_episode(client, episode_log):
    segmentation = episode_log['segmentation']
    task_instruction = episode_log['task_instruction']
    reasoning_text = episode_log['cosmos_reasoning_proxy_text']

    manual_review_reasons = []
    if not segmentation['segmentation_valid']:
        manual_review_reasons.append('segmentation_invalid')

    reasoning_result = judge_reasoning(client, task_instruction, reasoning_text)
    if reasoning_result['manual_review_needed']:
        manual_review_reasons.append('reasoning_hallucinated_or_underspecified')

    phase_classifications = {}
    for phase_name, phase_result in episode_log['phase_results'].items():
        execution_result = score_execution_phase(phase_result, segmentation)
        if execution_result['manual_review_ambiguous_minor']:
            manual_review_reasons.append(f'{phase_name}_ambiguous_execution')
        if execution_result['early_step_count'] < MIN_COMPARISON_STEPS:
            manual_review_reasons.append(f'{phase_name}_too_few_comparison_steps')

        reasoning_match_for_phase = bool(reasoning_result['phases_found'].get(phase_name, False))
        classification = classify_2x2(reasoning_match_for_phase, execution_result['overall_tier'])

        phase_classifications[phase_name] = {
            'reasoning_match': reasoning_match_for_phase,
            'execution_tier': execution_result['overall_tier'],
            'active_side': execution_result['active_side'],
            'execution_groups': execution_result['groups'],
            'full_horizon_groups': execution_result['full_horizon_groups'],
            'early_step_count': execution_result['early_step_count'],
            'full_horizon_step_count': phase_result.get('compared_steps', 0),
            'classification': classification,
        }

    return {
        'task_folder': episode_log['task_folder'],
        'episode_index': episode_log['episode_index'],
        'task_instruction': task_instruction,
        'segmentation_valid': segmentation['segmentation_valid'],
        'reasoning': reasoning_result,
        'phase_classifications': phase_classifications,
        'manual_review_reasons': sorted(set(manual_review_reasons)),
    }

print('Classification + episode orchestration ready.')

## 16. Run scoring across all logged episodes + write results

Reads every `episode_*.json` in `RAW_LOG_DIR`, scores it, and writes both a full JSON
(`results/scored_episodes.json`) and a flat per-phase CSV (`results/scored_phases.csv`)
to Drive — the CSV is the easiest thing to open and eyeball for the manual spot-check
step that comes after this.

In [ ]:
import csv

RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

log_files = sorted(f for f in os.listdir(RAW_LOG_DIR) if f.startswith('episode_') and f.endswith('.json'))
if not log_files:
    raise RuntimeError(f'No episode_*.json files found in {RAW_LOG_DIR} — run Section 10 first.')

scored_episodes = []
for fname in log_files:
    with open(os.path.join(RAW_LOG_DIR, fname)) as f:
        episode_log = json.load(f)
    print(f'Scoring {fname}...')
    scored_episodes.append(score_episode(anthropic_client, episode_log))

with open(os.path.join(RESULTS_DIR, 'scored_episodes.json'), 'w') as f:
    json.dump(scored_episodes, f, indent=2)

# Two of the four manual_review_reasons tags apply to the whole episode
# (segmentation_invalid, reasoning_hallucinated_or_underspecified); the other two are
# per-phase (f'{phase}_ambiguous_execution', f'{phase}_too_few_comparison_steps'). A
# phase row's needs_manual_review should reflect only reasons that actually apply to
# THAT phase -- not every reason anywhere in the episode, which would mark a clean
# 'reach' row as needing review just because 'transport' in the same episode was flagged.
EPISODE_WIDE_REVIEW_REASONS = {'segmentation_invalid', 'reasoning_hallucinated_or_underspecified'}

csv_path = os.path.join(RESULTS_DIR, 'scored_phases.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['task_folder', 'episode_index', 'phase', 'active_side', 'reasoning_match',
                      'execution_tier', 'early_step_count', 'full_horizon_tier', 'full_horizon_step_count',
                      'classification', 'needs_manual_review'])
    for ep in scored_episodes:
        episode_wide_flagged = any(r in EPISODE_WIDE_REVIEW_REASONS for r in ep['manual_review_reasons'])
        for phase_name, pc in ep['phase_classifications'].items():
            # Recompute the full-chunk equivalent of the primary-side verdict, purely
            # for side-by-side comparison in the CSV -- doesn't affect classification.
            active_side = pc['active_side']
            primary_keys = {f'{active_side}_arm', f'{active_side}_hand',
                             f'{active_side}_wrist_eef_9d', 'waist'}
            full_tier = _combine_tiers(_tiers_for_keys(pc['full_horizon_groups'], primary_keys))
            phase_flagged = (f'{phase_name}_ambiguous_execution' in ep['manual_review_reasons']
                              or f'{phase_name}_too_few_comparison_steps' in ep['manual_review_reasons'])
            needs_review = episode_wide_flagged or phase_flagged
            writer.writerow([ep['task_folder'], ep['episode_index'], phase_name, active_side,
                              pc['reasoning_match'], pc['execution_tier'], pc['early_step_count'],
                              full_tier, pc['full_horizon_step_count'],
                              pc['classification'], needs_review])

counts = {}
total_phases = 0
n_phases_flagged = 0
for ep in scored_episodes:
    episode_wide_flagged = any(r in EPISODE_WIDE_REVIEW_REASONS for r in ep['manual_review_reasons'])
    for phase_name, pc in ep['phase_classifications'].items():
        counts[pc['classification']] = counts.get(pc['classification'], 0) + 1
        total_phases += 1
        phase_flagged = (f'{phase_name}_ambiguous_execution' in ep['manual_review_reasons']
                          or f'{phase_name}_too_few_comparison_steps' in ep['manual_review_reasons'])
        if episode_wide_flagged or phase_flagged:
            n_phases_flagged += 1
n_episodes_flagged = sum(1 for ep in scored_episodes if ep['manual_review_reasons'])

print(f'\nScored {len(scored_episodes)} episodes, {total_phases} phase observations.')
print(f'{n_episodes_flagged} episodes have at least one flagged phase; '
      f'{n_phases_flagged}/{total_phases} individual phases need manual review.')
for label, count in sorted(counts.items(), key=lambda kv: -kv[1]):
    pct = 100.0 * count / total_phases if total_phases else 0.0
    print(f'  {label:28s} {count:4d}  ({pct:.1f}%)')

save_status('scoring_pass_1_scored', 'done')
print(f'\nWrote {csv_path} and scored_episodes.json to {RESULTS_DIR}')

## 17. Extract frame-exact images for a manual spot-check

Saves individual frames as PNGs around a flagged phase's start frame, read directly from
the already-downloaded episode video via sequential decode (frame-exact, same approach as
Section 8's `build_observation` fix -- not a browser-scrubbed approximation). Self-
contained: only needs Section 0 (Drive mount, for `PROJECT_DIR`/`RAW_LOG_DIR`) to have
run, not the GPU/model-loading sections. See `docs/manual_spot_check.md` for the full
walkthrough of how to use this alongside the raw log's reasoning text and the scored
execution numbers to judge whether a flagged phase's verdict looks right.

In [ ]:
import cv2

SPOT_CHECK_DIR = os.path.join(PROJECT_DIR, 'spot_check_frames')
os.makedirs(SPOT_CHECK_DIR, exist_ok=True)


def extract_phase_frames(episode_index, phase_name, context_frames=3, early_window=10):
    """Save frame-exact PNGs spanning a phase's scored window (start_frame, plus a few
    frames of context on each side) for manual spot-checking. Also prints the task
    instruction and Cosmos-Reason2's reasoning trace for the same episode, so everything
    needed for one spot-check is in one place."""
    log_path = os.path.join(RAW_LOG_DIR, f'episode_{episode_index:06d}.json')
    with open(log_path) as f:
        episode_log = json.load(f)

    task_folder = episode_log['task_folder']
    phase_result = episode_log['phase_results'][phase_name]
    start_frame = phase_result['start_frame']
    n_steps = min(phase_result.get('compared_steps', early_window), early_window)

    video_path = os.path.join(
        PROJECT_DIR, 'data', 'g1_teleop_subset', task_folder,
        'videos', 'chunk-000', 'observation.images.ego_view',
        f'episode_{episode_index:06d}.mp4',
    )
    if not os.path.exists(video_path):
        raise FileNotFoundError(
            f'Video not found at {video_path} -- the dataset must be downloaded (Section '
            f'5) at least once in this project, even if not in the current session.'
        )

    first_frame = max(0, start_frame - context_frames)
    last_frame = start_frame + n_steps + context_frames

    cap = cv2.VideoCapture(video_path)
    saved_paths = []
    for idx in range(last_frame + 1):
        ret, frame = cap.read()
        if not ret:
            break
        if idx < first_frame:
            continue
        out_path = os.path.join(
            SPOT_CHECK_DIR, f'ep{episode_index:06d}_{phase_name}_frame{idx:04d}.png',
        )
        cv2.imwrite(out_path, frame)
        saved_paths.append(out_path)
    cap.release()

    print(f'Episode {episode_index}, phase "{phase_name}": scored window is frames '
          f'{start_frame}-{start_frame + n_steps}; saved frames {first_frame}-'
          f'{first_frame + len(saved_paths) - 1} (with {context_frames}-frame padding) '
          f'to {SPOT_CHECK_DIR}')
    print(f'Task instruction: "{episode_log["task_instruction"]}"')
    print(f'Cosmos-Reason2 reasoning trace:\n{episode_log["cosmos_reasoning_proxy_text"]}')
    return saved_paths


# Example: episode 0's transport phase (flagged intent_lost_in_handoff in the last run).
# Change episode_index / phase_name and re-run for any other flagged phase.
extract_phase_frames(episode_index=0, phase_name='transport')

## 18. Failure-taxonomy analysis

Aggregates `scored_episodes.json` into the summary tables the paper draft (see
`docs/paper_draft.md`) is built from: the overall reasoning x execution classification
breakdown, the breakdown by sub-goal phase, which measured dimension (arm/hand/wrist
position/wrist rotation/waist) most often drives a failure or minor-deviation verdict,
and the reasoning-stage failure categories. Reads `results/scored_episodes.json` --
works regardless of how many episodes have been scored, so re-run this any time after a
new batch finishes to refresh the numbers.

**Interpretation caveat that must travel with every number this cell prints**: this
measures deviation from ONE recorded human demonstration per episode, not physical task
success. A phase scored `intent_lost_in_handoff` means GR00T's predicted trajectory
diverged from what the human teleoperator did next -- not that the robot would have
failed to physically complete the task. There may be multiple valid solution paths; this
rubric can only detect divergence from the one path that was recorded.

In [ ]:
import json
from collections import Counter, defaultdict

with open(os.path.join(RESULTS_DIR, 'scored_episodes.json')) as f:
    scored_episodes = json.load(f)

classification_counts = Counter()
phase_classification = defaultdict(Counter)
for ep in scored_episodes:
    for phase_name, pc in ep['phase_classifications'].items():
        classification_counts[pc['classification']] += 1
        phase_classification[phase_name][pc['classification']] += 1

total = sum(classification_counts.values())
print(f'=== Overall classification ({len(scored_episodes)} episodes, {total} phase observations) ===')
for label, count in classification_counts.most_common():
    print(f'  {label:24s} {count:4d}  ({100*count/total:.1f}%)')

print('\n=== By phase ===')
for phase_name in ['reach', 'transport', 'retreat']:
    counts = phase_classification[phase_name]
    phase_total = sum(counts.values())
    if phase_total == 0:
        continue
    print(f'{phase_name}:')
    for label, count in counts.most_common():
        print(f'  {label:24s} {count:4d}  ({100*count/phase_total:.1f}%)')

driving_group_counts = Counter()
for ep in scored_episodes:
    for phase_name, pc in ep['phase_classifications'].items():
        if pc['execution_tier'] not in ('failure', 'minor_deviation'):
            continue
        active = pc['active_side']
        groups = pc['execution_groups']
        for key in [f'{active}_arm', f'{active}_hand', f'{active}_wrist_eef_9d', 'waist']:
            g = groups.get(key)
            if g is None:
                continue
            if key.endswith('wrist_eef_9d'):
                for dim, tier_key in [('position', 'position_tier'), ('rotation', 'rotation_tier')]:
                    if g[tier_key] in ('failure', 'minor_deviation'):
                        driving_group_counts[f'wrist_{dim}'] += 1
            else:
                if g.get('group_tier') in ('failure', 'minor_deviation'):
                    label = 'hand' if key.endswith('hand') else ('arm' if key.endswith('arm') else 'waist')
                    driving_group_counts[label] += 1

print('\n=== Which measured dimension drives failure/minor_deviation (can be >1 per phase) ===')
for label, count in driving_group_counts.most_common():
    print(f'  {label:20s} {count:4d}')

reasoning_failure_counts = Counter()
for ep in scored_episodes:
    for cat in ep['reasoning']['failure_categories']:
        reasoning_failure_counts[cat] += 1
print(f'\n=== Reasoning-stage failure categories (episode-level, {len(scored_episodes)} episodes) ===')
for label, count in reasoning_failure_counts.most_common():
    print(f'  {label:28s} {count:4d} / {len(scored_episodes)} episodes')

taxonomy_summary = {
    'n_episodes': len(scored_episodes),
    'n_phase_observations': total,
    'classification_counts': dict(classification_counts),
    'phase_classification': {k: dict(v) for k, v in phase_classification.items()},
    'driving_group_counts': dict(driving_group_counts),
    'reasoning_failure_counts': dict(reasoning_failure_counts),
}
with open(os.path.join(RESULTS_DIR, 'taxonomy_summary.json'), 'w') as f:
    json.dump(taxonomy_summary, f, indent=2)
print(f'\nWrote {os.path.join(RESULTS_DIR, "taxonomy_summary.json")}')